In [1]:
import pandas as pd
import altair as alt

In [2]:
df = pd.read_csv('mba_decision_dataset.csv')
df_sample = df.sample(2000, random_state=42)

In [3]:
df_melted = df_sample.melt(
    id_vars=['Gender'], 
    value_vars=['Annual Salary (Before MBA)', 'Expected Post-MBA Salary'], 
    var_name='Salary Type', 
    value_name='Salary'
)

df_melted['Salary Type'] = df_melted['Salary Type'].replace({
    'Annual Salary (Before MBA)': 'Before MBA',
    'Expected Post-MBA Salary': 'After MBA'
})


male_select = alt.param(name="male_filter", bind=alt.binding_checkbox(name="Male"), value=True)
female_select = alt.param(name="female_filter", bind=alt.binding_checkbox(name="Female"), value=True)
other_select = alt.param(name="other_filter", bind=alt.binding_checkbox(name="Other"), value=True)

# Boxplot Chart
boxplot = alt.Chart(df_melted).mark_boxplot(size=50,color='blue').encode(
    x=alt.X('Salary Type:N', title='MBA Stage', sort=['Before MBA', 'After MBA']),
    y=alt.Y('Salary:Q', title='Salary ($)', scale=alt.Scale(domain=[0, 200000])),
    column=alt.Column('Gender:N', title='Gender'),
    tooltip=['Salary Type', 'Salary', 'Gender']
).properties(
    title="Salary Before & After MBA by Gender",
    width=200,
    height=600
).add_params(
    male_select, female_select, other_select
).transform_filter(
    (alt.datum.Gender == 'Male') & male_select |
    (alt.datum.Gender == 'Female') & female_select |
    (alt.datum.Gender == 'Other') & other_select
)

boxplot

alt.Chart(...)

In [4]:
df_mba_counts = df.groupby('Current Job Title').agg(
    total_count=('Decided to Pursue MBA?', 'count'),
    mba_yes_count=('Decided to Pursue MBA?', lambda x: (x == 'Yes').sum())
).reset_index()

df_mba_counts['mba_percentage'] = (df_mba_counts['mba_yes_count'] / df_mba_counts['total_count']) * 100

df_mba_counts = df_mba_counts.sort_values(by='mba_percentage', ascending=False).head(15)

bar_chart = alt.Chart(df_mba_counts).mark_bar(color='blue').encode(
    x=alt.X('Current Job Title:N', title='Pre-MBA Job', sort='-y', axis=alt.Axis(labelAngle=45)),
    y=alt.Y('mba_percentage:Q', title='% of People Who Pursued MBA', scale=alt.Scale(domain=[0, 100]))
).properties(
    title="Percentage of People Pursuing MBA by Pre-MBA Job",
    width=800,
    height=500
)

bar_chart

alt.Chart(...)

In [5]:
boxplot.save('pre-and-post-mba_salary_by_gender_boxplot.html')
# bar_chart.save('pre-mba_job_barplot.html')

In [6]:
set(df_melted['Gender'].values)

{'Female', 'Male', 'Other'}